# Trinity Admission Data Preparation <br>


The goal of this data preparation project is to ready the data for constructing a classification model to determine whether an accepted applicant will decide to attend the University, based on information collected regarding the University’s recently accepted applicants, ranging in entry term from Fall 2017, Fall 2018, Fall 2019, Fall 2020, and Fall 2021.  Your final data set should be ready for modeling. 

This script demonstrates how to clean some typical variables for the train dataset and the cleaning requiremens for all variables. 

The train dataset is a subset of the original dataset, which is used to train the model to understand the relationships between variables. Then the trained model will predict the target variable using predictors in the test dataset.

You need to handle all the columns/variables that are not processed in this scirpt following the intrsuctions in the comments.


In [ ]:
import pandas as pd
import numpy as np
#Read in TU.csv
TU = pd.read_csv("TU.csv")
# pd.set_option('display.max_columns', None)
TU.info()

In [ ]:
# Display all columns in one output
TU.columns

In [ ]:
# Divide the dataframe into training and test datasets
# In this course, you will only work on the variables in the training set
# You will need to clean the test set following the methods used in this script, when you work on modeling in later chapter.

TUtrain=TU[TU['train-test']=='train']
TUtest=TU[TU['train-test']=='test']

In [ ]:
#Column1 - ID

#Check NAs
TUtrain['ID'].isna().sum()
#No NA,so no cleaning is required. But ID will be removed in the modeling stage. Why?
TUtrain=TUtrain.drop('ID',axis='columns')

#Column2 - Entry Term Application

#Check NAs
print(TUtrain['Entry Term (Application)'].isna().sum())
#No NA.
TUtrain['Entry Term (Application)'].unique()
#No irregular categories.


#Column 3 - Admit Type

#Check NAs
print(TUtrain['Admit Type'].isna().sum())
#No NA.
print(TUtrain['Admit Type'].unique())
#No irregular categories.

#Since the data set only has first years (i.e.,only one category), 
# Admit.Type should be removed.
print(TUtrain['Admit Type'])
TUtrain=TUtrain.drop('Admit Type',axis='columns')
TUtrain.columns

#Column 4 - Permanent Postal
print(TUtrain['Permanent Postal'].isna().sum())
#105 NAs.
TUtrain['Permanent Postal'].unique()
#However, the column "Permanent.Geomarket" had already provided needed information
#regarding the postal codes of different states. Therefore, this column might be
#redundant, and we might just use "Permanent.Geomarket"
#Therefore, let's remove this column and there is no need to handle the missing values
print(TUtrain['Permanent Postal'])
TUtrain=TUtrain.drop('Permanent Postal',axis='columns')
TUtrain.columns

#Column 5 - Permanent Country
#0 NA.
print(TUtrain['Permanent Country'].isna().sum())
#No irregular categories.
TUtrain['Permanent Country'].unique()

#Column6 - Sex
print(TUtrain['Sex'].isna().sum())
#No NA.
print(TUtrain['Sex'].unique())
#No irregular categories.


#Column7 - Ethnicity
print(TUtrain['Ethnicity'].isna().sum())
#158 NAs.
print(TUtrain['Ethnicity'].unique())
#No irregular categories.
# It is fair to replace NAs with "Not Specified" as we do not have other columns for inter-field checking.

TUtrain['Ethnicity'].fillna("Not specified",inplace=True)
TUtrain['Ethnicity'].isna().sum()

#Column8 - Race
print(TUtrain['Race'].isna().sum())
#389 NAs.
print(TUtrain['Race'].unique())

#No irregular categories.
#Impute NAs with "Not specified", similar to what we do for Ethnicity.
TUtrain['Race'].fillna("Not specified", inplace=True)

#The current classification of Race is too detailed, which leads to very 
#low frequencies for some categories.Let's take a look at the value frequencies.
TUtrain['Race'].value_counts()

#So let's combine some of the categories because a category with a small number of cases won't have 
#a significant effect on the target variable in the modeling stage.

#Group labels with a low number of cases (< 100 cases) into "Others".
# first create a list for races with counts less than 100
races_counts =TUtrain['Race'].value_counts()
races_less_than_100 =races_counts[races_counts<100].index.tolist()
races_less_than_100

TUtrain['Race'] = \
TUtrain['Race'].apply(lambda x: 'Others' if x in races_less_than_100 else x)
TUtrain['Race'].value_counts()

#Column 9 - Religion
# print # of NAs
print(TUtrain['Religion'].isna().sum())

# print unique values for Religion
print(TUtrain['Religion'].unique())

#Impute NAs with "Not specified", similar to what we do for Race.
TUtrain['Religion'].fillna("Not specified", inplace=True)

#The current classification of Race is too detailed, which leads to very 
#low frequencies for some categories.Print the value frequencies.
religion_counts =TUtrain['Religion'].value_counts()
religion_counts

#Religion has lots of categories, with some categories having a very small number of cases. 
#Let's combine similar levels into one level( for example:['Bible Churches','Christian Reformed','Christian Scientist','Church of Christ','Church of God'] )
TUtrain['Religion'].apply(lambda x: 'Christian' if x in ['Bible Churches','Christian Reformed','Christian Scientist','Church of Christ','Church of God'] else x)

#then combine levels with less than 100 cases into "Other" because a level accounting for lower than 1% 
#of training set is very unlikely to have a significant effect on the target variable.
religion_less_than_100 =religion_counts[religion_counts<100].index.tolist()
TUtrain['Religion']= \
TUtrain['Religion'].apply(lambda x: 'Others' if x in religion_less_than_100 else x)
TUtrain['Religion'].value_counts()


#Column 10 - First_Source Origin First Source Date
print(TUtrain['First_Source Origin First Source Date'].isna().sum())
#No NAs.

#convert to date format
TUtrain['First_Source Origin First Source Date']=\
pd.to_datetime(TUtrain['First_Source Origin First Source Date'],format='mixed')
TUtrain['First_Source Origin First Source Date']

print(TUtrain['First_Source Origin First Source Date'].isna().sum())

#Column11 - Inquiry Date
print(TUtrain['Inquiry Date'].isna().sum())
#3181 NAs.

#convert to date format
TUtrain['Inquiry Date']= pd.to_datetime(TUtrain['Inquiry Date'], errors='coerce')


#Column 12 - Submitted
print(TUtrain['Submitted'].isna().sum())
#No NAs.

#convert to date format
TUtrain['Submitted']=pd.to_datetime(TUtrain['Submitted'], errors='coerce',format='mixed')

print(TUtrain['Submitted'].isna().sum())



#Column10-12
print(TUtrain['First_Source Origin First Source Date'].isna().sum())


#After viewing Column10-12, it would be interesting to see
#whether the differences between submission date and First_Source date
#the differences between submission date and inquiry date affect the response variable.
#So let's calculate the time difference between submission date and first_source date.
TUtrain['Submit_FirstSource'] = \
(TUtrain['Submitted']-TUtrain['First_Source Origin First Source Date']).astype('timedelta64[us]')
TUtrain['Submit_Inquiry']= \
(TUtrain['Submitted']-TUtrain['Inquiry Date']).astype('timedelta64[us]')
print(TUtrain['Submit_FirstSource'].isna().sum())

TUtrain['Submit_FirstSource'] = TUtrain['Submit_FirstSource'].dt.days
TUtrain['Submit_Inquiry'] = TUtrain['Submit_Inquiry'].dt.days

#There are NAs in Inquiry.Date, 
#thus leading to NAs in Submit_Inquiry.
#Impute NAs in Submit_Inquiry with mean.

TUtrain['Submit_Inquiry'].fillna(TUtrain['Submit_Inquiry'].mean(),inplace=True)
TUtrain['Submit_Inquiry'].isna().sum()

#Remove Column10-12 after you created new variables above.   
TUtrain=TUtrain.drop('First_Source Origin First Source Date',axis='columns')
TUtrain=TUtrain.drop('Inquiry Date',axis='columns')
TUtrain=TUtrain.drop('Submitted',axis='columns')



#Column13 - Application.Source

#Check NAs
print(TUtrain['Application Source'].isna().sum())

#Check irregular categories. If there are irregular values, you need to handle them.
print(TUtrain['Application Source'].unique())

#Column14 - Decision Plan

#Check NAs
print(TUtrain['Decision Plan'].isna().sum())

#Check irregular categories. If there are irregular values, you need to handle them.
print(TUtrain['Decision Plan'].unique())

#replace Early Action II and Early action I with Early Action
TUtrain['Decision Plan'] = TUtrain['Decision Plan'].apply(lambda x: 'Early Action' if x in ['Early Action I', 'Early Action II'] else x)
print(TUtrain['Decision Plan'].unique())

#Column15 - Staff.Assigned.Name
#Based on variable description, this variable might not be useful and provide
#insightful information in the modeling.
#Also, some staffs already left Trinity.
#So remove this variable
TUtrain=TUtrain.drop('Staff Assigned Name',axis='columns')


#Column16 - Legacy
print(TUtrain['Legacy'].isna().sum())
#No NAs.
print(TUtrain['Legacy'].unique())
#No irregular categories.
#Impute NAs with "No Legacy"
TUtrain['Legacy'].fillna("No Legacy",inplace=True)
TUtrain['Legacy'].isna().sum()

#Legacy has many options, leading some options to having only a small number of cases.
#Let's group all the options into 3 categories ('Legacy',"No Legacy", "Legacy, Opt Out") 
#so that each category has the chance to affect the response variable.
TUtrain['Legacy']=\
TUtrain['Legacy'].apply(lambda x: 'Legacy, Opt Out' if x not in ['Legacy','No Legacy'] else x)

#Column17 - Athlete
# print # NAs.
print(TUtrain['Athlete'].isna().sum())
# print unique value counts.
print(TUtrain['Athlete'].unique())

#Impute NAs with "Non-Athlete"
TUtrain['Athlete'].fillna("Non-Athlete",inplace=True)
TUtrain['Athlete'].isna().sum()

#Similar to Legacy, Athlete has many categories with a few cases.
#Group all options into three categories: 
#Athlete, Non-Athlete, and Athlete, Opt Out.
TUtrain['Athlete']=\
TUtrain['Athlete'].apply(lambda x: 'Opt Out' if x not in ['Athlete','Non-Athlete'] else x)

#Column18 - Sport 1 Sport

# print # NAs.
print(TUtrain['Sport 1 Sport'].isna().sum())
# print unique value counts.
print(TUtrain['Sport 1 Sport'].unique())

#Impute NAs with "No Sport"
TUtrain['Sport 1 Sport'].fillna("No Sport",inplace=True)
TUtrain['Sport 1 Sport'].isna().sum()
#Group sport men and sport women into one group
#so that each group has sufficient cases to have an impact on the response.
TUtrain['Sport 1 Sport']=TUtrain['Sport 1 Sport'].apply(lambda x: x.replace(' Men', '').replace(' Women', ''))
print(TUtrain['Sport 1 Sport'].unique())

#Column19 - Sport 1 Rating
# print # NAs.
print(TUtrain['Sport 1 Rating'].isna().sum())
# print unique value counts.
print(TUtrain['Sport 1 Rating'].unique())

#Impute NAs with "No Sport"
TUtrain['Sport 1 Rating'].fillna("No Sport",inplace=True)
TUtrain['Sport 1 Rating'].isna().sum()

#Column20 - Sport 2 Sport

# print # NAs.
print(TUtrain['Sport 2 Sport'].isna().sum())
# print unique value counts.
print(TUtrain['Sport 2 Sport'].unique())

#Impute NAs with "No 2ndSport"
TUtrain['Sport 2 Sport'].fillna("No 2ndSport",inplace=True)
TUtrain['Sport 2 Sport'].isna().sum()

#The number of cases for each sport type is very small (< about 1% of the data set).
#It's better to group all options into 2 categories: 2ndSport vs. No 2ndSport.
TUtrain['Sport 2 Sport']=TUtrain['Sport 2 Sport'].apply(lambda x: '2ndSport' if x not in ['No 2ndSport'] else x)
print(TUtrain['Sport 2 Sport'].unique())

#Column21 - Sport 2 Rating
print(TUtrain['Sport 2 Rating'].isna().sum())
#9957 NAs.
print(TUtrain['Sport 2 Rating'].unique())
#Only 43 out of 10000 observations are rated, which is less than 0.5% of the data set!
#Sport.2.Rating will not have much impact on the target.
#Remove it in the modeling stage.
TUtrain.drop('Sport 2 Rating',axis='columns', inplace=True)


#Column22 - Sport 3 Sport

# print # NAs.
print(TUtrain['Sport 3 Sport'].isna().sum())
# print unique value counts.
print(TUtrain['Sport 3 Sport'].unique())
#impute NAs with "No 3rdSport".
TUtrain['Sport 3 Sport'].fillna("No 3rdSport",inplace=True)

#The number of cases for each sport type is very small (< 0.5% of the data set).
#It's better to group all options into 2 categories: 3rdSport vs. No 3rdSport.
TUtrain['Sport 3 Sport']=TUtrain['Sport 3 Sport'].apply(lambda x: '3rdSport' if x not in ['No 3rdSport'] else x)

#Column23 - Sport.3.Rating

print(TUtrain['Sport 3 Rating'].isna().sum())
#9998 NAs.
print(TUtrain['Sport 3 Rating'].unique())
#No questionable category.
#Only 2 out of 10000 observations are rated, which will not provide much insightful
#information. Therefore,remove this column
TUtrain.drop('Sport 3 Rating',axis='columns', inplace=True)


#Column24 - Academic Interest 1

#check NAs
print(TUtrain['Academic Interest 1'].isna().sum())
# 4 NAs.
print(TUtrain['Academic Interest 1'].unique())



# Step1: Most of the NAs for Academic.Interest.1 have a value for Academic.Interest.2
#We may assign the corresponding values in Academic.Interest.2 
#to NAs in Academic.Interest.1 if Academic.Interest.2 has a value.

# When update values in a subset of dataframes, 
# Try using .loc[row_indexer,col_indexer] = value instead to avoid chained indexing issue

for index,row in TUtrain.iterrows():
    if pd.isna(row['Academic Interest 1']):
        #print(index,row['Academic Interest 1'],row['Academic Interest 2'])
        TUtrain.loc[index,'Academic Interest 1']=TUtrain.loc[index,'Academic Interest 2']

    
# Step2:For the remaining NAs in Academic.Interest.1, assign Undecided.
TUtrain['Academic Interest 1'].fillna('Undecided',inplace=True)


#  Step3: Group Business related options into "Business".
# find all business related labels and group them into business
TUtrain['Academic Interest 1'] = \
TUtrain['Academic Interest 1'].apply(lambda x: 'Business' if x in['Business','Finance','Business - Communication Management','Business - Accounting',
                                                                 'Business - Sport Management','Business - Management','Business - Marketing',
                                                                 'Business - International Business','Business Analytics & Technology',
                                                                 'Business - Management Information Systems','Business Legal Studies'] else x)


# step 4: Group majors with a low number of cases (< 100 cases) into "Others".
# first create a list for majors with counts less than 100
AI1_counts =TUtrain['Academic Interest 1'].value_counts()
AI1_less_than_100 =AI1_counts[AI1_counts<100].index.tolist()

TUtrain['Academic Interest 1'] = \
TUtrain['Academic Interest 1'].apply(lambda x: 'Others' if x in AI1_less_than_100 else x)
TUtrain['Academic Interest 1'].value_counts()

#Column25 - Academic.Interest.2

#Check NAs
print(TUtrain['Academic Interest 2'].isna().sum())


#Replace repeated academic interest 2 with Undecided, which means academic interest 2==academic interest 1
TUtrain.loc[TUtrain['Academic Interest 1']==TUtrain['Academic Interest 2'], 'Academic Interest 2'] = 'Undecided'

#then replace NAs Undecided
TUtrain['Academic Interest 2'].fillna("Undecided",inplace=True)

#Group Business related options into "Business".
TUtrain['Academic Interest 2'] = \
TUtrain['Academic Interest 2'].apply(lambda x: 'Business' if x in['Business','Finance','Business - Communication Management','Business - Accounting',
                                                                 'Business - Sport Management','Business - Management','Business - Marketing',
                                                                 'Business - International Business','Business Analytics & Technology',
                                                                 'Business - Management Information Systems','Business Legal Studies'] else x)


#Group options with a low number of cases (< 100 cases) into "Others".
AI2_counts =TUtrain['Academic Interest 2'].value_counts()
AI2_less_than_100 =AI2_counts[AI2_counts<100].index.tolist()

TUtrain['Academic Interest 2'] = \
TUtrain['Academic Interest 2'].apply(lambda x: 'Others' if x in AI2_less_than_100 else x)
TUtrain['Academic Interest 2'].value_counts()



#Column26 - First_Source Origin First Source Summary

# print # NAs.
print(TUtrain['First_Source Origin First Source Summary'].isna().sum())
# print unique value counts.
print(TUtrain['First_Source Origin First Source Summary'].unique())
#Similar to Academic.Interest.2, group options with a low number of cases (< 100) into "Other Sources".
First_Source_counts =TUtrain['First_Source Origin First Source Summary'].value_counts()
First_Source_less_than_100 =First_Source_counts[First_Source_counts<100].index.tolist()

TUtrain['First_Source Origin First Source Summary'] = \
TUtrain['First_Source Origin First Source Summary'].apply(lambda x: 'Others' if x in First_Source_less_than_100 else x)
TUtrain['First_Source Origin First Source Summary'].value_counts()

#Column27 - Total Event Participation


#Check NAs.
print(TUtrain['Total Event Participation'].isna().sum())

#check irregular values
print(TUtrain['Total Event Participation'].unique())

#3, 4, 5 combined accounts for < 1% of the data set.
#Compared to the number of cases in 0, 1, and 2, the number of cases
#in 3, 4, and 5 won't be very useful in predicting the response.
#So group 3, 4, and 5 into "2 or more".
TUtrain['Total Event Participation'] = TUtrain['Total Event Participation'].astype(int)
TUtrain['Total Event Participation'] = TUtrain['Total Event Participation'].apply(lambda x: '2 or more' if x >= 2 else x)
TUtrain['Total Event Participation'] = TUtrain['Total Event Participation'].astype(str)
#Column28 - Count of Campus Visits

# print # NAs.
print(TUtrain['Count of Campus Visits'].isna().sum())
# print unique value counts.
print(TUtrain['Count of Campus Visits'].unique())
# group 5, 6, and 8 into '4 or more'.
TUtrain['Count of Campus Visits'] = TUtrain['Count of Campus Visits'].astype(int)
TUtrain['Count of Campus Visits'] = TUtrain['Count of Campus Visits'].apply(lambda x: '5 or more' if x >= 5 else x)
TUtrain['Count of Campus Visits'] = TUtrain['Count of Campus Visits'].astype(str)
#Column29 - School #1 Organization Category
print(TUtrain['School #1 Organization Category'].isna().sum())
#25  NAs.
print(TUtrain['School #1 Organization Category'].value_counts())
#Only 8 cases belong to College but 9967 cases belong to High School.
#Remove this variable.
TUtrain.drop('School #1 Organization Category', axis='columns', inplace=True)

#Column30 - School 1 Code
print(TUtrain['School 1 Code'].isna().sum())
#7842 NAs.
print(TUtrain['School 1 Code'].unique())
#School Code will not matter much to produce insightful information.
#Additionally, there are 7842 missing values.
#so remove this column in the modeling stage.
TUtrain.drop('School 1 Code', axis='columns', inplace=True)

#Column31 - School 1 Class Rank (Numeric)
print(TU.loc[TU['train-test']=='train','School 1 Class Rank (Numeric)'].isna().sum())
#5357 NAs.

# Column32 - School 1 Class Size (Numeric)

print(TUtrain['School 1 Class Size (Numeric)'].isna().sum())

#5357 NAs.
#Percentage rank can more accurately reflect a student's academic performance than numeric rank. 

#Create a New Column - School 1 Top Percent in Class

TUtrain['School 1 Top Percent in Class'] =\
100 *(TUtrain['School 1 Class Rank (Numeric)']/TUtrain['School 1 Class Size (Numeric)'])
TUtrain.drop('School 1 Class Size (Numeric)', axis='columns', inplace=True)
TUtrain.drop('School 1 Class Rank (Numeric)', axis='columns', inplace=True)
TUtrain['School 1 Top Percent in Class'].isna().sum()

# #Impute the 5357 NAs based on Academic.Index column. 

# #Since we need to handle NAs in School 1 Top Percent in Class
# according to Academic.Index, first let's see whether Academic Index needs to be cleaned.
print(TUtrain['Academic Index'].isna().sum())
#829 NAs.
print(TUtrain['Academic Index'].value_counts())
#No questionable level.
#Impute 829 NAs with the most common level.
TUtrain['Academic Index'].fillna(3,inplace=True)
TUtrain['Academic Index'].unique()
#No missing values in Academic Index now.

#calculate school 1 top percent in class for each academic index group
grouped=TUtrain.groupby('Academic Index')
grouped
average=grouped.mean('School 1 Top Percent in Class')
average['School 1 Top Percent in Class']

#Impute missing values in 'School 1 Top Percent in Class' based on Academic Index group average
for i,row in TUtrain.iterrows():

    if (row['Academic Index']== 1.0) & (pd.isna(row['School 1 Top Percent in Class'])):
        TUtrain.loc[i,'School 1 Top Percent in Class']= average['School 1 Top Percent in Class'][1.0]
    elif (row['Academic Index']== 2.0) & (pd.isna(row['School 1 Top Percent in Class'])):
            TUtrain.loc[i,'School 1 Top Percent in Class']= average['School 1 Top Percent in Class'][2.0]
    elif (row['Academic Index']== 3.0) & (pd.isna(row['School 1 Top Percent in Class'])):
            TUtrain.loc[i,'School 1 Top Percent in Class']= average['School 1 Top Percent in Class'][3.0]
    elif (row['Academic Index']== 4.0) & (pd.isna(row['School 1 Top Percent in Class'])):
            TUtrain.loc[i,'School 1 Top Percent in Class']= average['School 1 Top Percent in Class'][4.0]
    elif (row['Academic Index']== 5.0) & (pd.isna(row['School 1 Top Percent in Class'])):
            TUtrain.loc[i,'School 1 Top Percent in Class']= average['School 1 Top Percent in Class'][5.0]
print(TUtrain['Academic Index'].isna().sum())

#Column33 - School 1 GPA

#Remove this variable in the modeling stage
#because School.1.GPA.Recalculated is more accurate.
TUtrain.drop('School 1 GPA', axis='columns', inplace=True)


#Column34 - School 1 GPA Scale
#Remove this variable in the modeling stage as it is irrelevant.
TUtrain.drop('School 1 GPA Scale', axis='columns', inplace=True)




#Column35 - School 1 GPA Recalculated

#Check NAs

print(TUtrain['School 1 GPA Recalculated'].isna().sum())
#0 NAs.

TUtrain['School 1 GPA Recalculated'].skew()
#Check skewness

# if the skewness score is below -1 or above 1, the variable is high skewed
#if the skewness score is positive, it means it is right tail skew
# if the skewness score is negative, it means it is left tail skew
#Since it is moderately skewed, and it is understandable for the left skewness as 
#a lot of students got into Trinity with a high GPA (almost 4.0), it is unnecessary to do transformation.
# Some modeling methods requires variables following a normal distribution, therefore you need to transform the skewed data
# before inputting it into the model, such as liner regression analysis.

#Column36 - School 2 Class Rank (Numeric)
#Check NAs
print(TUtrain['School 2 Class Rank (Numeric)'].isna().sum())
#Should we keep or remove this variable. Justify your decision in comments.
#There are no values so we should drop this column.
TUtrain.drop('School 2 Class Rank (Numeric)', axis='columns', inplace=True)


#Column37 - School 2 Class Size (Numeric)
print(TUtrain['School 2 Class Size (Numeric)'].isna().sum())
#Check NAs

#Should we keep or remove this variable. Justify your decision in comments.
#There are no values so we should drop this column.
TUtrain.drop('School 2 Class Size (Numeric)', axis='columns', inplace=True)

#Column38 - School 2 GPA

#Check NAs
print(TUtrain['School 2 GPA'].isna().sum())
#Should we keep or remove this variable. Justify your decision in comments.
#There are no values so we should drop this column.
TUtrain.drop('School 2 GPA', axis='columns', inplace=True)


#Column39 - School 2 GPA Scale
#Check NAs
print(TUtrain['School 2 GPA Scale'].isna().sum())
#Should we keep or remove this variable. Justify your decision in comments.
#There are no values so we should drop this column.
TUtrain.drop('School 2 GPA Scale', axis='columns', inplace=True)

#Column40 - School 2 GPA Recalculated
#Check NAs
print(TUtrain['School 2 GPA Recalculated'].isna().sum())
#Should we keep or remove this variable. Justify your decision in comments.
#There are no values so we should drop this column.
TUtrain.drop('School 2 GPA Recalculated', axis='columns', inplace=True)

#Column41 - School 3 Class Rank (Numeric)
#Check NAs
print(TUtrain['School 3 Class Rank (Numeric)'].isna().sum())
#Should we keep or remove this variable. Justify your decision in comments.
#There are no values so we should drop this column.
TUtrain.drop('School 3 Class Rank (Numeric)', axis='columns', inplace=True)

#Column42 - School 3 Class Size (Numeric)
#Check NAs
print(TUtrain['School 3 Class Size (Numeric)'].isna().sum())
#Should we keep or remove this variable. Justify your decision in comments.
#There are no values so we should drop this column.
TUtrain.drop('School 3 Class Size (Numeric)', axis='columns', inplace=True)


#Column43 - School 3 GPA
#Check NAs
print(TUtrain['School 3 GPA'].isna().sum())
#Should we keep or remove this variable. Justify your decision in comments.
#There are no values so we should drop this column.
TUtrain.drop('School 3 GPA', axis='columns', inplace=True)


#Column44 - School 3 GPA Scale
#Check NAs
print(TUtrain['School 3 GPA Scale'].isna().sum())
#Should we keep or remove this variable. Justify your decision in comments.
#There are no values so we should drop this column.
TUtrain.drop('School 3 GPA Scale', axis='columns', inplace=True)


#Column45 - School 3 GPA Recalculated
#Check NAs
print(TUtrain['School 3 GPA Recalculated'].isna().sum())
#Should we keep or remove this variable. Justify your decision in comments.
#There are no values so we should drop this column.
TUtrain.drop('School 3 GPA Recalculated', axis='columns', inplace=True)

#Column46 ACT Composite

# print # NAs.
print(TUtrain['ACT Composite'].isna().sum())
# print unique value counts.
print(TUtrain['ACT Composite'].value_counts())
#Replace missing ACT scores with SAT Concordance scores. 
#Convert 'SAT R Evidence-Based Reading and Writing Section + Math Section scores' into ACT based on the ACT-SAT concordance table pdf;
#Hint: create a dictionary to convert SAT into ACT
SAT_ACT = {1570: 36, 1530: 35, 1490: 34, 1450: 33, 1420: 32, 1390: 31, 1360: 30, 
           1330: 29, 1300: 28, 1260: 27, 1230: 26, 1200: 25, 1160: 24, 1130: 23, 
           1100: 22, 1060: 21, 1030: 20, 990: 19, 960: 18, 920: 17, 880: 16, 
           830: 15, 780: 14, 730: 13, 690: 12, 650: 11, 620: 10, 590: 9}

# Replace missing ACT scores with converted SAT scores.
def convert_sat_to_act(sat_score):
    for sat_cutoff, act_score in sorted(SAT_ACT.items(), reverse=True):
        if sat_score >= sat_cutoff:
            return act_score
    return np.nan

# Apply conversion for missing ACT scores using SAT scores
TUtrain.loc[TUtrain['ACT Composite'].isna(), 'ACT Composite'] = (
    TUtrain.loc[TUtrain['ACT Composite'].isna(), 'SAT R Evidence-Based Reading and Writing Section + Math Section']
    .apply(convert_sat_to_act)
)

#Third, impute the rest missing ACTs with the mean of ACTs.
TUtrain['ACT Composite'].fillna(TUtrain['ACT Composite'].mean(),inplace=True)

print(TUtrain['ACT Composite'].mean())

#Column47 ACT English

#Check NAs 
print(TUtrain['ACT English'].isna().sum())
# Since ACT Composite is already a good indicator for ACT scores generally,scores on each section will not matter much to make analyses.
#Remove this variable.
TUtrain.drop('ACT English', axis='columns', inplace=True)

#Column48 ACT Reading
#Check NAs 
print(TUtrain['ACT Reading'].isna().sum())
# Since ACT Composite is already a good indicator for ACT scores generally,scores on each section will not matter much to make analyses.
#Remove this variable.
TUtrain.drop('ACT Reading', axis='columns', inplace=True)

#Column50 ACT Math
#Check NAs 
print(TUtrain['ACT Math'].isna().sum())

# Since ACT Composite is already a good indicator for ACT scores generally,scores on each section will not matter much to make analyses.
#Remove this variable.
TUtrain.drop('ACT Math', axis='columns', inplace=True)

#Column51 ACT Science Reasoning
#Check NAs 
print(TUtrain['ACT Science Reasoning'].isna().sum())


# Since ACT Composite is already a good indicator for ACT scores generally,scores on each section will not matter much to make analyses.
#Remove this variable.
TUtrain.drop('ACT Science Reasoning', axis='columns', inplace=True)

#Column52 ACT Writing
#??? NAs 
print(TUtrain['ACT Writing'].isna().sum())
# Since ACT Composite is already a good indicator for ACT scores generally,scores on each section will not matter much to make analyses.
#Remove this variable.
TUtrain.drop('ACT Writing', axis='columns', inplace=True)

#Column53 SAT I CR + M
#Check NAs 
print(TUtrain['SAT I CR + M'].isna().sum())

# Since ACT Composite is already a good indicator for ACT scores generally,scores on each section will not matter much to make analyses.
#Remove this variable.
TUtrain.drop('SAT I CR + M', axis='columns', inplace=True)

#Column54 SAT R Evidence-Based Reading and Writing Section + Math Section
TUtrain.drop('SAT R Evidence-Based Reading and Writing Section + Math Section', axis='columns', inplace=True)


# This column is used to generate ATC concordance scores.
# No further processing needed.

#Column55 Permanent Geomarket
#First, replace the missing values with the most frequent geo market value.
print(TUtrain['Permanent Geomarket'].isna().sum())
print(TUtrain['Permanent Geomarket'].value_counts())
#Second, group geomarket values into different regions. Refer to the region .csv for grouping.

# Get the most frequent geomarket safely
geomarket_mode = TUtrain['Permanent Geomarket'].dropna().mode()
geomarket_mode = geomarket_mode[0] if not geomarket_mode.empty else "Unknown"

# Fill missing values with the most frequent geomarket
TUtrain['Permanent Geomarket'].fillna(geomarket_mode, inplace=True)

# Load Regions.csv and set 'State' as index
Region = pd.read_csv('Regions.csv')
Region.set_index('State', inplace=True)

# State abbreviation dictionary
State_Abrv = {
    "AK": "Alaska", "AL": "Alabama", "AR": "Arkansas", "AZ": "Arizona", "CA": "California",
    "CO": "Colorado", "CT": "Connecticut", "DE": "Delaware", "FL": "Florida", "GA": "Georgia",
    "HI": "Hawaii", "IA": "Iowa", "ID": "Idaho", "IL": "Illinois", "IN": "Indiana", "KS": "Kansas",
    "KY": "Kentucky", "LA": "Louisiana", "MA": "Massachusetts", "MD": "Maryland", "ME": "Maine",
    "MI": "Michigan", "MN": "Minnesota", "MO": "Missouri", "MS": "Mississippi", "MT": "Montana",
    "NC": "North Carolina", "ND": "North Dakota", "NE": "Nebraska", "NH": "New Hampshire",
    "NJ": "New Jersey", "NM": "New Mexico", "NV": "Nevada", "NY": "New York", "OH": "Ohio",
    "OK": "Oklahoma", "OR": "Oregon", "PA": "Pennsylvania", "RI": "Rhode Island", "SC": "South Carolina",
    "SD": "South Dakota", "TN": "Tennessee", "TX": "Texas", "UT": "Utah", "VA": "Virginia",
    "VT": "Vermont", "WA": "Washington", "WI": "Wisconsin", "WV": "West Virginia", "WY": "Wyoming",
    "DC": "D.C.", "INT": "International"
}

# Extract state abbreviations from 'Permanent Geomarket' (before the '-')
TUtrain['Permanent Geomarket'] = TUtrain['Permanent Geomarket'].str.split('-').str[0]

# Map the extracted state code to the full state name
TUtrain['Permanent Geomarket'] = TUtrain['Permanent Geomarket'].map(State_Abrv)

# Fill remaining NaNs with the most frequent geomarket again (in case some didn't match)
TUtrain['Permanent Geomarket'].fillna(geomarket_mode, inplace=True)

# Map the full state name to its region using Regions.csv
TUtrain['Permanent Geomarket'] = TUtrain['Permanent Geomarket'].map(Region['Region'])

# Fill remaining N/As from international and unknown with other
TUtrain['Permanent Geomarket'].fillna("Other (International/Unknown)", inplace=True)
# Print updated value counts
print(TUtrain['Permanent Geomarket'].value_counts())


#Column56 Citizenship Status

# This column is used for inter-field checking so keep it

#Column57 Academic Index

# This column is used for inter-field checking so keep it

#Column58 Intend to Apply for Financial Aid?
print(TUtrain['Intend to Apply for Financial Aid?'].isna().sum())

#Handling missing values. Justify your choice.
#It doesn't make sense to use mean for a binary choice so use mode
TUtrain['Intend to Apply for Financial Aid?'].fillna(1,inplace=True)
print(TUtrain['Intend to Apply for Financial Aid?'].isna().sum())

#Column59 Merit Award

#Refer to the 'Merit Award Code.csv) for grouping. 
#Recategorize all the levels into fewer levels, Justify your grouping policy in comments
#Merit_Code=pd.read_csv('Admissions Data_Merit Award Codes.csv')

Merit_Code = {
    'International': [
        'I10', 'I12', 'I12.5', 'I15', 'I17', 'I18', 'I19', 'I20', 
        'I21', 'I24', 'I25', 'I26', 'I27', 'I28', 'I30', 'I32', 
        'I33', 'I35', 'I38', 'I5', 'I9', 'I0'
    ],
    'Full Ride': [
        'SEM', 'TTS'
    ],
    'Full Tuition': [
        'X0', 'Y0'
    ],
    'Domestic': [
        'D12.5', 'Z0', 'I0', 'M24', 'P17', 'P17', 'TT9', 'TTS', 'T21'
    ]
}

code_to_category = {}
for category, codes in Merit_Code.items():
    for code in codes:
        code_to_category[code] = category

# Replace the Merit Award codes with the corresponding categories
TUtrain['Merit Award'] = TUtrain['Merit Award'].map(code_to_category)

#Fill remaining NAs with No Merit
TUtrain['Merit Award'].fillna("None", inplace=True)

print(TUtrain['Merit Award'].value_counts())

#Column60 SAT Concordance Score (of SAT R)
print(TUtrain['SAT Concordance Score (of SAT R)'].isna().sum())
#Remove this variable.Justify why you remove it.
#Since ACT Composite is already a good indicator for ACT scores generally,scores on each section will not matter much to make analyses.
TUtrain.drop('SAT Concordance Score (of SAT R)', axis='columns', inplace=True)

#Column61 ACT Concordance Score (of SAT R)
print(TUtrain['ACT Concordance Score (of SAT R)'].isna().sum())
#Remove this variable.Justify why you remove it.
#Since ACT Composite is already a good indicator for ACT scores generally,scores on each section will not matter much to make analyses.
TUtrain.drop('ACT Concordance Score (of SAT R)', axis='columns', inplace=True)

#Column62 ACT Concordance Score (of SAT)
print(TUtrain['ACT Concordance Score (of SAT)'].isna().sum())
#Remove this variable.Justify why you remove it.
#Almost no values
TUtrain.drop('ACT Concordance Score (of SAT)', axis='columns', inplace=True)

#Column63 Test Optional
print(TUtrain['Test Optional'].isna().sum())
#Remove this variable.Justify why you remove it.
#Since ACT Composite is already a good indicator for ACT scores generally,scores on each section will not matter much to make analyses.
TUtrain.drop('Test Optional', axis='columns', inplace=True)

#Column64 SAT I Critical Reading
print(TUtrain['SAT I Critical Reading'].isna().sum())
#Remove this variable.Justify why you remove it.
#Almost no data
TUtrain.drop('SAT I Critical Reading', axis='columns', inplace=True)

#Column65 SAT I Math
print(TUtrain['SAT I Math'].isna().sum())
#Remove this variable.Justify why you remove it.
#Almost no data
TUtrain.drop('SAT I Math', axis='columns', inplace=True)

#Column66 SAT I Writing
print(TUtrain['SAT I Writing'].isna().sum())
#Remove this variable.Justify why you remove it.
#Almost no data
TUtrain.drop('SAT I Writing', axis='columns', inplace=True)

#Column67 SAT R Evidence-Based Reading and Writing Section
print(TUtrain['SAT R Evidence-Based Reading and Writing Section'].isna().sum())
#Remove this variable.Justify why you remove it.
#Since ACT Composite is already a good indicator for ACT scores generally,scores on each section will not matter much to make analyses.
TUtrain.drop('SAT R Evidence-Based Reading and Writing Section', axis='columns', inplace=True)

#Column68 SAT R Math Section
print(TUtrain['SAT R Math Section'].isna().sum())
#Remove this variable.Justify why you remove it.
#Since ACT Composite is already a good indicator for ACT scores generally,scores on each section will not matter much to make analyses.
TUtrain.drop('SAT R Math Section', axis='columns', inplace=True)

#Column69 Decision

# This would be your dependent variable in the classification model so keep it.

# After you clean all variables, output the cleaned dataframe to a csv file
# the csv file can be found in your current working directory

TUtrain2=TUtrain
TUtrain.to_csv('cleaneddftrain.csv')

# Compare the your table structure with the screenshot in the submission box
# Make sure all primary predictors and target variables do not have any missing values and only have
# regualr and correct values.

In [ ]:
TUtrain.info()

In [ ]:
#Column1 - ID

#Check NAs
TUtest['ID'].isna().sum()
#No NA,so no cleaning is required. But ID will be removed in the modeling stage. Why?
TUtest=TUtest.drop('ID',axis='columns')
#Column2 - Entry Term Application

#Check NAs
print(TUtest['Entry Term (Application)'].isna().sum())
#No NA.
TUtest['Entry Term (Application)'].unique()
#No irregular categories.


#Column 3 - Admit Type

#Check NAs
print(TUtest['Admit Type'].isna().sum())
#No NA.
print(TUtest['Admit Type'].unique())
#No irregular categories.

#Since the data set only has first years (i.e.,only one category), 
# Admit.Type should be removed.
print(TUtest['Admit Type'])
TUtest=TUtest.drop('Admit Type',axis='columns')
TUtest.columns

#Column 4 - Permanent Postal
print(TUtest['Permanent Postal'].isna().sum())
#105 NAs.
TUtest['Permanent Postal'].unique()
#However, the column "Permanent.Geomarket" had already provided needed information
#regarding the postal codes of different states. Therefore, this column might be
#redundant, and we might just use "Permanent.Geomarket"
#Therefore, let's remove this column and there is no need to handle the missing values
print(TUtest['Permanent Postal'])
TUtest=TUtest.drop('Permanent Postal',axis='columns')
TUtest.columns

#Column 5 - Permanent Country
#0 NA.
print(TUtest['Permanent Country'].isna().sum())
#No irregular categories.
TUtest['Permanent Country'].unique()

#Column6 - Sex
print(TUtest['Sex'].isna().sum())
#No NA.
print(TUtest['Sex'].unique())
#No irregular categories.


#Column7 - Ethnicity
print(TUtest['Ethnicity'].isna().sum())
#158 NAs.
print(TUtest['Ethnicity'].unique())
#No irregular categories.
# It is fair to replace NAs with "Not Specified" as we do not have other columns for inter-field checking.

TUtest['Ethnicity'].fillna("Not specified",inplace=True)
TUtest['Ethnicity'].isna().sum()

#Column8 - Race
print(TUtest['Race'].isna().sum())
#389 NAs.
print(TUtest['Race'].unique())

#No irregular categories.
#Impute NAs with "Not specified", similar to what we do for Ethnicity.
TUtest['Race'].fillna("Not specified", inplace=True)

#The current classification of Race is too detailed, which leads to very 
#low frequencies for some categories.Let's take a look at the value frequencies.
TUtest['Race'].value_counts()

#So let's combine some of the categories because a category with a small number of cases won't have 
#a significant effect on the target variable in the modeling stage.

#Group labels with a low number of cases (< 100 cases) into "Others".
# first create a list for races with counts less than 100

TUtest['Race'] = \
TUtest['Race'].apply(lambda x: 'Others' if x in races_less_than_100 else x)
TUtest['Race'].value_counts()

#Column 9 - Religion
# print # of NAs
print(TUtest['Religion'].isna().sum())

# print unique values for Religion
print(TUtest['Religion'].unique())

#Impute NAs with "Not specified", similar to what we do for Race.
TUtest['Religion'].fillna("Not specified", inplace=True)

#The current classification of Race is too detailed, which leads to very 
#low frequencies for some categories.Print the value frequencies.
religion_counts =TUtest['Religion'].value_counts()
religion_counts

#Religion has lots of categories, with some categories having a very small number of cases. 
#Let's combine similar levels into one level( for example:['Bible Churches','Christian Reformed','Christian Scientist','Church of Christ','Church of God'] )
TUtest['Religion'].apply(lambda x: 'Christian' if x in ['Bible Churches','Christian Reformed','Christian Scientist','Church of Christ','Church of God'] else x)

#then combine levels with less than 100 cases into "Other" because a level accounting for lower than 1% 
#of testing set is very unlikely to have a significant effect on the target variable.
TUtest['Religion']= \
TUtest['Religion'].apply(lambda x: 'Others' if x in religion_less_than_100 else x)
TUtest['Religion'].value_counts()


#Column 10 - First_Source Origin First Source Date
print(TUtest['First_Source Origin First Source Date'].isna().sum())
#No NAs.

#convert to date format
TUtest['First_Source Origin First Source Date']=\
pd.to_datetime(TUtest['First_Source Origin First Source Date'],format='mixed')
TUtest['First_Source Origin First Source Date']

print(TUtest['First_Source Origin First Source Date'].isna().sum())

#Column11 - Inquiry Date
print(TUtest['Inquiry Date'].isna().sum())
#3181 NAs.

#convert to date format
TUtest['Inquiry Date']= pd.to_datetime(TUtest['Inquiry Date'], errors='coerce')


#Column 12 - Submitted
print(TUtest['Submitted'].isna().sum())
#No NAs.

#convert to date format
TUtest['Submitted']=pd.to_datetime(TUtest['Submitted'], errors='coerce',format='mixed')

print(TUtest['Submitted'].isna().sum())



#Column10-12
print(TUtest['First_Source Origin First Source Date'].isna().sum())


#After viewing Column10-12, it would be interesting to see
#whether the differences between submission date and First_Source date
#the differences between submission date and inquiry date affect the response variable.
#So let's calculate the time difference between submission date and first_source date.
TUtest['Submit_FirstSource'] = \
(TUtest['Submitted']-TUtest['First_Source Origin First Source Date']).astype('timedelta64[us]')
TUtest['Submit_Inquiry']= \
(TUtest['Submitted']-TUtest['Inquiry Date']).astype('timedelta64[us]')
print(TUtest['Submit_FirstSource'].isna().sum())

TUtest['Submit_FirstSource'] = TUtest['Submit_FirstSource'].dt.days
TUtest['Submit_Inquiry'] = TUtest['Submit_Inquiry'].dt.days
#There are NAs in Inquiry.Date, 
#thus leading to NAs in Submit_Inquiry.
#Impute NAs in Submit_Inquiry with mean.

TUtest['Submit_Inquiry'].fillna(TUtest['Submit_Inquiry'].mean(),inplace=True)
TUtest['Submit_Inquiry'].isna().sum()

#Remove Column10-12 after you created new variables above.   
TUtest=TUtest.drop('First_Source Origin First Source Date',axis='columns')
TUtest=TUtest.drop('Inquiry Date',axis='columns')
TUtest=TUtest.drop('Submitted',axis='columns')



#Column13 - Application.Source

#Check NAs
print(TUtest['Application Source'].isna().sum())

#Check irregular categories. If there are irregular values, you need to handle them.
print(TUtest['Application Source'].unique())

#Column14 - Decision Plan

#Check NAs
print(TUtest['Decision Plan'].isna().sum())

#Check irregular categories. If there are irregular values, you need to handle them.
print(TUtest['Decision Plan'].unique())

#replace Early Action II and Early action I with Early Action
TUtest['Decision Plan'] = TUtest['Decision Plan'].apply(lambda x: 'Early Action' if x in ['Early Action I', 'Early Action II'] else x)
print(TUtest['Decision Plan'].unique())

#Column15 - Staff.Assigned.Name
#Based on variable description, this variable might not be useful and provide
#insightful information in the modeling.
#Also, some staffs already left Trinity.
#So remove this variable
TUtest=TUtest.drop('Staff Assigned Name',axis='columns')


#Column16 - Legacy
print(TUtest['Legacy'].isna().sum())
#No NAs.
print(TUtest['Legacy'].unique())
#No irregular categories.
#Impute NAs with "No Legacy"
TUtest['Legacy'].fillna("No Legacy",inplace=True)
TUtest['Legacy'].isna().sum()

#Legacy has many options, leading some options to having only a small number of cases.
#Let's group all the options into 3 categories ('Legacy',"No Legacy", "Legacy, Opt Out") 
#so that each category has the chance to affect the response variable.
TUtest['Legacy']=\
TUtest['Legacy'].apply(lambda x: 'Legacy, Opt Out' if x not in ['Legacy','No Legacy'] else x)

#Column17 - Athlete
# print # NAs.
print(TUtest['Athlete'].isna().sum())
# print unique value counts.
print(TUtest['Athlete'].unique())

#Impute NAs with "Non-Athlete"
TUtest['Athlete'].fillna("Non-Athlete",inplace=True)
TUtest['Athlete'].isna().sum()

#Similar to Legacy, Athlete has many categories with a few cases.
#Group all options into three categories: 
#Athlete, Non-Athlete, and Athlete, Opt Out.
TUtest['Athlete']=\
TUtest['Athlete'].apply(lambda x: 'Opt Out' if x not in ['Athlete','Non-Athlete'] else x)

#Column18 - Sport 1 Sport

# print # NAs.
print(TUtest['Sport 1 Sport'].isna().sum())
# print unique value counts.
print(TUtest['Sport 1 Sport'].unique())

#Impute NAs with "No Sport"
TUtest['Sport 1 Sport'].fillna("No Sport",inplace=True)
TUtest['Sport 1 Sport'].isna().sum()
#Group sport men and sport women into one group
#so that each group has sufficient cases to have an impact on the response.
TUtest['Sport 1 Sport']=TUtest['Sport 1 Sport'].apply(lambda x: x.replace(' Men', '').replace(' Women', ''))
print(TUtest['Sport 1 Sport'].unique())

#Column19 - Sport 1 Rating
# print # NAs.
print(TUtest['Sport 1 Rating'].isna().sum())
# print unique value counts.
print(TUtest['Sport 1 Rating'].unique())

#Impute NAs with "No Sport"
TUtest['Sport 1 Rating'].fillna("No Sport",inplace=True)
TUtest['Sport 1 Rating'].isna().sum()

#Column20 - Sport 2 Sport

# print # NAs.
print(TUtest['Sport 2 Sport'].isna().sum())
# print unique value counts.
print(TUtest['Sport 2 Sport'].unique())

#Impute NAs with "No 2ndSport"
TUtest['Sport 2 Sport'].fillna("No 2ndSport",inplace=True)
TUtest['Sport 2 Sport'].isna().sum()

#The number of cases for each sport type is very small (< about 1% of the data set).
#It's better to group all options into 2 categories: 2ndSport vs. No 2ndSport.
TUtest['Sport 2 Sport']=TUtest['Sport 2 Sport'].apply(lambda x: '2ndSport' if x not in ['No 2ndSport'] else x)
print(TUtest['Sport 2 Sport'].unique())

#Column21 - Sport 2 Rating
print(TUtest['Sport 2 Rating'].isna().sum())
#9957 NAs.
print(TUtest['Sport 2 Rating'].unique())
#Only 43 out of 10000 observations are rated, which is less than 0.5% of the data set!
#Sport.2.Rating will not have much impact on the target.
#Remove it in the modeling stage.
TUtest.drop('Sport 2 Rating',axis='columns', inplace=True)


#Column22 - Sport 3 Sport

# print # NAs.
print(TUtest['Sport 3 Sport'].isna().sum())
# print unique value counts.
print(TUtest['Sport 3 Sport'].unique())
#impute NAs with "No 3rdSport".
TUtest['Sport 3 Sport'].fillna("No 3rdSport",inplace=True)

#The number of cases for each sport type is very small (< 0.5% of the data set).
#It's better to group all options into 2 categories: 3rdSport vs. No 3rdSport.
TUtest['Sport 3 Sport']=TUtest['Sport 3 Sport'].apply(lambda x: '3rdSport' if x not in ['No 3rdSport'] else x)

#Column23 - Sport.3.Rating

print(TUtest['Sport 3 Rating'].isna().sum())
#9998 NAs.
print(TUtest['Sport 3 Rating'].unique())
#No questionable category.
#Only 2 out of 10000 observations are rated, which will not provide much insightful
#information. Therefore,remove this column
TUtest.drop('Sport 3 Rating',axis='columns', inplace=True)


#Column24 - Academic Interest 1

#check NAs
print(TUtest['Academic Interest 1'].isna().sum())
# 4 NAs.
print(TUtest['Academic Interest 1'].unique())



# Step1: Most of the NAs for Academic.Interest.1 have a value for Academic.Interest.2
#We may assign the corresponding values in Academic.Interest.2 
#to NAs in Academic.Interest.1 if Academic.Interest.2 has a value.

# When update values in a subset of dataframes, 
# Try using .loc[row_indexer,col_indexer] = value instead to avoid chained indexing issue

for index,row in TUtest.iterrows():
    if pd.isna(row['Academic Interest 1']):
        #print(index,row['Academic Interest 1'],row['Academic Interest 2'])
        TUtest.loc[index,'Academic Interest 1']=TUtest.loc[index,'Academic Interest 2']

    
# Step2:For the remaining NAs in Academic.Interest.1, assign Undecided.
TUtest['Academic Interest 1'].fillna('Undecided',inplace=True)


#  Step3: Group Business related options into "Business".
# find all business related labels and group them into business
TUtest['Academic Interest 1'] = \
TUtest['Academic Interest 1'].apply(lambda x: 'Business' if x in['Business','Finance','Business - Communication Management','Business - Accounting',
                                                                 'Business - Sport Management','Business - Management','Business - Marketing',
                                                                 'Business - International Business','Business Analytics & Technology',
                                                                 'Business - Management Information Systems','Business Legal Studies'] else x)


# step 4: Group majors with a low number of cases (< 100 cases) into "Others".
# first create a list for majors with counts less than 100

TUtest['Academic Interest 1'] = \
TUtest['Academic Interest 1'].apply(lambda x: 'Others' if x in AI1_less_than_100 else x)
TUtest['Academic Interest 1'].value_counts()

#Column25 - Academic.Interest.2

#Check NAs
print(TUtest['Academic Interest 2'].isna().sum())


#Replace repeated academic interest 2 with Undecided, which means academic interest 2==academic interest 1
TUtest.loc[TUtest['Academic Interest 1']==TUtest['Academic Interest 2'], 'Academic Interest 2'] = 'Undecided'

#then replace NAs Undecided
TUtest['Academic Interest 2'].fillna("Undecided",inplace=True)

#Group Business related options into "Business".
TUtest['Academic Interest 2'] = \
TUtest['Academic Interest 2'].apply(lambda x: 'Business' if x in['Business','Finance','Business - Communication Management','Business - Accounting',
                                                                 'Business - Sport Management','Business - Management','Business - Marketing',
                                                                 'Business - International Business','Business Analytics & Technology',
                                                                 'Business - Management Information Systems','Business Legal Studies'] else x)


#Group options with a low number of cases (< 100 cases) into "Others".

TUtest['Academic Interest 2'] = \
TUtest['Academic Interest 2'].apply(lambda x: 'Others' if x in AI2_less_than_100 else x)
TUtest['Academic Interest 2'].value_counts()



#Column26 - First_Source Origin First Source Summary

# print # NAs.
print(TUtest['First_Source Origin First Source Summary'].isna().sum())
# print unique value counts.
print(TUtest['First_Source Origin First Source Summary'].unique())
#Similar to Academic.Interest.2, group options with a low number of cases (< 100) into "Other Sources".

TUtest['First_Source Origin First Source Summary'] = \
TUtest['First_Source Origin First Source Summary'].apply(lambda x: 'Others' if x in First_Source_less_than_100 else x)
TUtest['First_Source Origin First Source Summary'].value_counts()

#Column27 - Total Event Participation


#Check NAs.
print(TUtest['Total Event Participation'].isna().sum())

#check irregular values
print(TUtest['Total Event Participation'].unique())

#3, 4, 5 combined accounts for < 1% of the data set.
#Compared to the number of cases in 0, 1, and 2, the number of cases
#in 3, 4, and 5 won't be very useful in predicting the response.
#So group 3, 4, and 5 into "2 or more".
TUtest['Total Event Participation'] = TUtest['Total Event Participation'].astype(int)
TUtest['Total Event Participation'] = TUtest['Total Event Participation'].apply(lambda x: '2 or more' if x >= 2 else x)
TUtest['Total Event Participation'] = TUtest['Total Event Participation'].astype(str)
#Column28 - Count of Campus Visits

# print # NAs.
print(TUtest['Count of Campus Visits'].isna().sum())
# print unique value counts.
print(TUtest['Count of Campus Visits'].unique())
# group 5, 6, and 8 into '4 or more'.
TUtest['Count of Campus Visits'] = TUtest['Count of Campus Visits'].astype(int)
TUtest['Count of Campus Visits'] = TUtest['Count of Campus Visits'].apply(lambda x: '5 or more' if x >= 5 else x)
TUtest['Count of Campus Visits'] = TUtest['Count of Campus Visits'].astype(str)
#Column29 - School #1 Organization Category
print(TUtest['School #1 Organization Category'].isna().sum())
#25  NAs.
print(TUtest['School #1 Organization Category'].value_counts())
#Only 8 cases belong to College but 9967 cases belong to High School.
#Remove this variable.
TUtest.drop('School #1 Organization Category', axis='columns', inplace=True)

#Column30 - School 1 Code
print(TUtest['School 1 Code'].isna().sum())
#7842 NAs.
print(TUtest['School 1 Code'].unique())
#School Code will not matter much to produce insightful information.
#Additionally, there are 7842 missing values.
#so remove this column in the modeling stage.
TUtest.drop('School 1 Code', axis='columns', inplace=True)

#Column31 - School 1 Class Rank (Numeric)
print(TU.loc[TU['train-test']=='test','School 1 Class Rank (Numeric)'].isna().sum())

#5357 NAs.

# Column32 - School 1 Class Size (Numeric)

print(TUtest['School 1 Class Size (Numeric)'].isna().sum())

#5357 NAs.
#Percentage rank can more accurately reflect a student's academic performance than numeric rank. 

#Create a New Column - School 1 Top Percent in Class

TUtest['School 1 Top Percent in Class'] =\
100 *(TUtest['School 1 Class Rank (Numeric)']/TUtest['School 1 Class Size (Numeric)'])
TUtest.drop('School 1 Class Size (Numeric)', axis='columns', inplace=True)
TUtest.drop('School 1 Class Rank (Numeric)', axis='columns', inplace=True)
TUtest['School 1 Top Percent in Class'].isna().sum()

# #Impute the 5357 NAs based on Academic.Index column. 

# #Since we need to handle NAs in School 1 Top Percent in Class
# according to Academic.Index, first let's see whether Academic Index needs to be cleaned.
print(TUtest['Academic Index'].isna().sum())
#829 NAs.
print(TUtest['Academic Index'].value_counts())
#No questionable level.
#Impute 829 NAs with the most common level.
TUtest['Academic Index'].fillna(3,inplace=True)
TUtest['Academic Index'].unique()
#No missing values in Academic Index now.

#calculate school 1 top percent in class for each academic index group
grouped=TUtest.groupby('Academic Index')
grouped
average=grouped.mean('School 1 Top Percent in Class')
average['School 1 Top Percent in Class']

#Impute missing values in 'School 1 Top Percent in Class' based on Academic Index group average
for i,row in TUtest.iterrows():

    if (row['Academic Index']== 1.0) & (pd.isna(row['School 1 Top Percent in Class'])):
        TUtest.loc[i,'School 1 Top Percent in Class']= average['School 1 Top Percent in Class'][1.0]
    elif (row['Academic Index']== 2.0) & (pd.isna(row['School 1 Top Percent in Class'])):
            TUtest.loc[i,'School 1 Top Percent in Class']= average['School 1 Top Percent in Class'][2.0]
    elif (row['Academic Index']== 3.0) & (pd.isna(row['School 1 Top Percent in Class'])):
            TUtest.loc[i,'School 1 Top Percent in Class']= average['School 1 Top Percent in Class'][3.0]
    elif (row['Academic Index']== 4.0) & (pd.isna(row['School 1 Top Percent in Class'])):
            TUtest.loc[i,'School 1 Top Percent in Class']= average['School 1 Top Percent in Class'][4.0]
    elif (row['Academic Index']== 5.0) & (pd.isna(row['School 1 Top Percent in Class'])):
            TUtest.loc[i,'School 1 Top Percent in Class']= average['School 1 Top Percent in Class'][5.0]
print(TUtest['Academic Index'].isna().sum())

#Column33 - School 1 GPA

#Remove this variable in the modeling stage
#because School.1.GPA.Recalculated is more accurate.
TUtest.drop('School 1 GPA', axis='columns', inplace=True)


#Column34 - School 1 GPA Scale
#Remove this variable in the modeling stage as it is irrelevant.
TUtest.drop('School 1 GPA Scale', axis='columns', inplace=True)




#Column35 - School 1 GPA Recalculated

#Check NAs

print(TUtest['School 1 GPA Recalculated'].isna().sum())
#0 NAs.

TUtest['School 1 GPA Recalculated'].skew()
#Check skewness

# if the skewness score is below -1 or above 1, the variable is high skewed
#if the skewness score is positive, it means it is right tail skew
# if the skewness score is negative, it means it is left tail skew
#Since it is moderately skewed, and it is understandable for the left skewness as 
#a lot of students got into Trinity with a high GPA (almost 4.0), it is unnecessary to do transformation.
# Some modeling methods requires variables following a normal distribution, therefore you need to transform the skewed data
# before inputting it into the model, such as liner regression analysis.

#Column36 - School 2 Class Rank (Numeric)
#Check NAs
print(TUtest['School 2 Class Rank (Numeric)'].isna().sum())
#Should we keep or remove this variable. Justify your decision in comments.
#There are no values so we should drop this column.
TUtest.drop('School 2 Class Rank (Numeric)', axis='columns', inplace=True)


#Column37 - School 2 Class Size (Numeric)
print(TUtest['School 2 Class Size (Numeric)'].isna().sum())
#Check NAs

#Should we keep or remove this variable. Justify your decision in comments.
#There are no values so we should drop this column.
TUtest.drop('School 2 Class Size (Numeric)', axis='columns', inplace=True)

#Column38 - School 2 GPA

#Check NAs
print(TUtest['School 2 GPA'].isna().sum())
#Should we keep or remove this variable. Justify your decision in comments.
#There are no values so we should drop this column.
TUtest.drop('School 2 GPA', axis='columns', inplace=True)


#Column39 - School 2 GPA Scale
#Check NAs
print(TUtest['School 2 GPA Scale'].isna().sum())
#Should we keep or remove this variable. Justify your decision in comments.
#There are no values so we should drop this column.
TUtest.drop('School 2 GPA Scale', axis='columns', inplace=True)

#Column40 - School 2 GPA Recalculated
#Check NAs
print(TUtest['School 2 GPA Recalculated'].isna().sum())
#Should we keep or remove this variable. Justify your decision in comments.
#There are no values so we should drop this column.
TUtest.drop('School 2 GPA Recalculated', axis='columns', inplace=True)

#Column41 - School 3 Class Rank (Numeric)
#Check NAs
print(TUtest['School 3 Class Rank (Numeric)'].isna().sum())
#Should we keep or remove this variable. Justify your decision in comments.
#There are no values so we should drop this column.
TUtest.drop('School 3 Class Rank (Numeric)', axis='columns', inplace=True)

#Column42 - School 3 Class Size (Numeric)
#Check NAs
print(TUtest['School 3 Class Size (Numeric)'].isna().sum())
#Should we keep or remove this variable. Justify your decision in comments.
#There are no values so we should drop this column.
TUtest.drop('School 3 Class Size (Numeric)', axis='columns', inplace=True)


#Column43 - School 3 GPA
#Check NAs
print(TUtest['School 3 GPA'].isna().sum())
#Should we keep or remove this variable. Justify your decision in comments.
#There are no values so we should drop this column.
TUtest.drop('School 3 GPA', axis='columns', inplace=True)


#Column44 - School 3 GPA Scale
#Check NAs
print(TUtest['School 3 GPA Scale'].isna().sum())
#Should we keep or remove this variable. Justify your decision in comments.
#There are no values so we should drop this column.
TUtest.drop('School 3 GPA Scale', axis='columns', inplace=True)


#Column45 - School 3 GPA Recalculated
#Check NAs
print(TUtest['School 3 GPA Recalculated'].isna().sum())
#Should we keep or remove this variable. Justify your decision in comments.
#There are no values so we should drop this column.
TUtest.drop('School 3 GPA Recalculated', axis='columns', inplace=True)

#Column46 ACT Composite

# print # NAs.
print(TUtest['ACT Composite'].isna().sum())
# print unique value counts.
print(TUtest['ACT Composite'].value_counts())
#Replace missing ACT scores with SAT Concordance scores. 
#Convert 'SAT R Evidence-Based Reading and Writing Section + Math Section scores' into ACT based on the ACT-SAT concordance table pdf;
#Hint: create a dictionary to convert SAT into ACT
SAT_ACT = {1570: 36, 1530: 35, 1490: 34, 1450: 33, 1420: 32, 1390: 31, 1360: 30, 
           1330: 29, 1300: 28, 1260: 27, 1230: 26, 1200: 25, 1160: 24, 1130: 23, 
           1100: 22, 1060: 21, 1030: 20, 990: 19, 960: 18, 920: 17, 880: 16, 
           830: 15, 780: 14, 730: 13, 690: 12, 650: 11, 620: 10, 590: 9}

# Replace missing ACT scores with converted SAT scores.
def convert_sat_to_act(sat_score):
    for sat_cutoff, act_score in sorted(SAT_ACT.items(), reverse=True):
        if sat_score >= sat_cutoff:
            return act_score
    return np.nan

# Apply conversion for missing ACT scores using SAT scores
TUtest.loc[TUtest['ACT Composite'].isna(), 'ACT Composite'] = (
    TUtest.loc[TUtest['ACT Composite'].isna(), 'SAT R Evidence-Based Reading and Writing Section + Math Section']
    .apply(convert_sat_to_act)
)

#Third, impute the rest missing ACTs with the mean of ACTs.
TUtest['ACT Composite'].fillna(TUtest['ACT Composite'].mean(),inplace=True)

print(TUtest['ACT Composite'].mean())

#Column47 ACT English

#Check NAs 
print(TUtest['ACT English'].isna().sum())
# Since ACT Composite is already a good indicator for ACT scores generally,scores on each section will not matter much to make analyses.
#Remove this variable.
TUtest.drop('ACT English', axis='columns', inplace=True)

#Column48 ACT Reading
#Check NAs 
print(TUtest['ACT Reading'].isna().sum())
# Since ACT Composite is already a good indicator for ACT scores generally,scores on each section will not matter much to make analyses.
#Remove this variable.
TUtest.drop('ACT Reading', axis='columns', inplace=True)

#Column50 ACT Math
#Check NAs 
print(TUtest['ACT Math'].isna().sum())

# Since ACT Composite is already a good indicator for ACT scores generally,scores on each section will not matter much to make analyses.
#Remove this variable.
TUtest.drop('ACT Math', axis='columns', inplace=True)

#Column51 ACT Science Reasoning
#Check NAs 
print(TUtest['ACT Science Reasoning'].isna().sum())


# Since ACT Composite is already a good indicator for ACT scores generally,scores on each section will not matter much to make analyses.
#Remove this variable.
TUtest.drop('ACT Science Reasoning', axis='columns', inplace=True)

#Column52 ACT Writing
#??? NAs 
print(TUtest['ACT Writing'].isna().sum())
# Since ACT Composite is already a good indicator for ACT scores generally,scores on each section will not matter much to make analyses.
#Remove this variable.
TUtest.drop('ACT Writing', axis='columns', inplace=True)

#Column53 SAT I CR + M
#Check NAs 
print(TUtest['SAT I CR + M'].isna().sum())

# Since ACT Composite is already a good indicator for ACT scores generally,scores on each section will not matter much to make analyses.
#Remove this variable.
TUtest.drop('SAT I CR + M', axis='columns', inplace=True)

#Column54 SAT R Evidence-Based Reading and Writing Section + Math Section
TUtest.drop('SAT R Evidence-Based Reading and Writing Section + Math Section', axis='columns', inplace=True)


# This column is used to generate ATC concordance scores.
# No further processing needed.

#Column55 Permanent Geomarket
#First, replace the missing values with the most frequent geo market value.
print(TUtest['Permanent Geomarket'].isna().sum())
print(TUtest['Permanent Geomarket'].value_counts())
#Second, group geomarket values into different regions. Refer to the region .csv for grouping.

# Get the most frequent geomarket safely
geomarket_mode = TUtest['Permanent Geomarket'].dropna().mode()
geomarket_mode = geomarket_mode[0] if not geomarket_mode.empty else "Unknown"

# Fill missing values with the most frequent geomarket
TUtest['Permanent Geomarket'].fillna(geomarket_mode, inplace=True)

# Load Regions.csv and set 'State' as index
Region = pd.read_csv('Regions.csv')
Region.set_index('State', inplace=True)

# State abbreviation dictionary
State_Abrv = {
    "AK": "Alaska", "AL": "Alabama", "AR": "Arkansas", "AZ": "Arizona", "CA": "California",
    "CO": "Colorado", "CT": "Connecticut", "DE": "Delaware", "FL": "Florida", "GA": "Georgia",
    "HI": "Hawaii", "IA": "Iowa", "ID": "Idaho", "IL": "Illinois", "IN": "Indiana", "KS": "Kansas",
    "KY": "Kentucky", "LA": "Louisiana", "MA": "Massachusetts", "MD": "Maryland", "ME": "Maine",
    "MI": "Michigan", "MN": "Minnesota", "MO": "Missouri", "MS": "Mississippi", "MT": "Montana",
    "NC": "North Carolina", "ND": "North Dakota", "NE": "Nebraska", "NH": "New Hampshire",
    "NJ": "New Jersey", "NM": "New Mexico", "NV": "Nevada", "NY": "New York", "OH": "Ohio",
    "OK": "Oklahoma", "OR": "Oregon", "PA": "Pennsylvania", "RI": "Rhode Island", "SC": "South Carolina",
    "SD": "South Dakota", "TN": "Tennessee", "TX": "Texas", "UT": "Utah", "VA": "Virginia",
    "VT": "Vermont", "WA": "Washington", "WI": "Wisconsin", "WV": "West Virginia", "WY": "Wyoming",
    "DC": "D.C.", "INT": "International"
}

# Extract state abbreviations from 'Permanent Geomarket' (before the '-')
TUtest['Permanent Geomarket'] = TUtest['Permanent Geomarket'].str.split('-').str[0]

# Map the extracted state code to the full state name
TUtest['Permanent Geomarket'] = TUtest['Permanent Geomarket'].map(State_Abrv)

# Fill remaining NaNs with the most frequent geomarket again (in case some didn't match)
TUtest['Permanent Geomarket'].fillna(geomarket_mode, inplace=True)

# Map the full state name to its region using Regions.csv
TUtest['Permanent Geomarket'] = TUtest['Permanent Geomarket'].map(Region['Region'])

# Fill remaining N/As from international and unknown with other
TUtest['Permanent Geomarket'].fillna("Other (International/Unknown)", inplace=True)
# Print updated value counts
print(TUtest['Permanent Geomarket'].value_counts())


#Column56 Citizenship Status

# This column is used for inter-field checking so keep it

#Column57 Academic Index

# This column is used for inter-field checking so keep it

#Column58 Intend to Apply for Financial Aid?
print(TUtest['Intend to Apply for Financial Aid?'].isna().sum())

#Handling missing values. Justify your choice.
#It doesn't make sense to use mean for a binary choice so use mode
TUtest['Intend to Apply for Financial Aid?'].fillna(1,inplace=True)
print(TUtest['Intend to Apply for Financial Aid?'].isna().sum())

#Column59 Merit Award

#Refer to the 'Merit Award Code.csv) for grouping. 
#Recategorize all the levels into fewer levels, Justify your grouping policy in comments
#Merit_Code=pd.read_csv('Admissions Data_Merit Award Codes.csv')

Merit_Code = {
    'International': [
        'I10', 'I12', 'I12.5', 'I15', 'I17', 'I18', 'I19', 'I20', 
        'I21', 'I24', 'I25', 'I26', 'I27', 'I28', 'I30', 'I32', 
        'I33', 'I35', 'I38', 'I5', 'I9', 'I0'
    ],
    'Full Ride': [
        'SEM', 'TTS'
    ],
    'Full Tuition': [
        'X0', 'Y0'
    ],
    'Domestic': [
        'D12.5', 'Z0', 'I0', 'M24', 'P17', 'P17', 'TT9', 'TTS', 'T21'
    ]
}

code_to_category = {}
for category, codes in Merit_Code.items():
    for code in codes:
        code_to_category[code] = category

# Replace the Merit Award codes with the corresponding categories
TUtest['Merit Award'] = TUtest['Merit Award'].map(code_to_category)

#Fill remaining NAs with No Merit
TUtest['Merit Award'].fillna("None", inplace=True)

print(TUtest['Merit Award'].value_counts())

#Column60 SAT Concordance Score (of SAT R)
print(TUtest['SAT Concordance Score (of SAT R)'].isna().sum())
#Remove this variable.Justify why you remove it.
#Since ACT Composite is already a good indicator for ACT scores generally,scores on each section will not matter much to make analyses.
TUtest.drop('SAT Concordance Score (of SAT R)', axis='columns', inplace=True)

#Column61 ACT Concordance Score (of SAT R)
print(TUtest['ACT Concordance Score (of SAT R)'].isna().sum())
#Remove this variable.Justify why you remove it.
#Since ACT Composite is already a good indicator for ACT scores generally,scores on each section will not matter much to make analyses.
TUtest.drop('ACT Concordance Score (of SAT R)', axis='columns', inplace=True)

#Column62 ACT Concordance Score (of SAT)
print(TUtest['ACT Concordance Score (of SAT)'].isna().sum())
#Remove this variable.Justify why you remove it.
#Almost no values
TUtest.drop('ACT Concordance Score (of SAT)', axis='columns', inplace=True)

#Column63 Test Optional
print(TUtest['Test Optional'].isna().sum())
#Remove this variable.Justify why you remove it.
#Since ACT Composite is already a good indicator for ACT scores generally,scores on each section will not matter much to make analyses.
TUtest.drop('Test Optional', axis='columns', inplace=True)

#Column64 SAT I Critical Reading
print(TUtest['SAT I Critical Reading'].isna().sum())
#Remove this variable.Justify why you remove it.
#Almost no data
TUtest.drop('SAT I Critical Reading', axis='columns', inplace=True)

#Column65 SAT I Math
print(TUtest['SAT I Math'].isna().sum())
#Remove this variable.Justify why you remove it.
#Almost no data
TUtest.drop('SAT I Math', axis='columns', inplace=True)

#Column66 SAT I Writing
print(TUtest['SAT I Writing'].isna().sum())
#Remove this variable.Justify why you remove it.
#Almost no data
TUtest.drop('SAT I Writing', axis='columns', inplace=True)

#Column67 SAT R Evidence-Based Reading and Writing Section
print(TUtest['SAT R Evidence-Based Reading and Writing Section'].isna().sum())
#Remove this variable.Justify why you remove it.
#Since ACT Composite is already a good indicator for ACT scores generally,scores on each section will not matter much to make analyses.
TUtest.drop('SAT R Evidence-Based Reading and Writing Section', axis='columns', inplace=True)

#Column68 SAT R Math Section
print(TUtest['SAT R Math Section'].isna().sum())
#Remove this variable.Justify why you remove it.
#Since ACT Composite is already a good indicator for ACT scores generally,scores on each section will not matter much to make analyses.
TUtest.drop('SAT R Math Section', axis='columns', inplace=True)

#Column69 Decision

# This would be your dependent variable in the classification model so keep it.

# After you clean all variables, output the cleaned dataframe to a csv file
# the csv file can be found in your current working directory

TUtest2=TUtest
TUtest.to_csv('cleaneddftest.csv')

# Compare the your table structure with the screenshot in the submission box
# Make sure all primary predictors and target variables do not have any missing values and only have
# regualr and correct values.

In [ ]:
TUtest.info()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import cohen_kappa_score
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import PartialDependenceDisplay
import matplotlib.pyplot as plt

In [ ]:
def encode (df):
    for col in df.columns:
        if df[col].dtype == 'object' or df[col].dtype == 'category':
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col])
    return df
TUtrain = encode(TUtrain)
TUtest= encode(TUtest)

In [ ]:
y_train= TUtrain['Decision']
x_train = TUtrain.drop(columns='Decision')
y_test=TUtest['Decision']
x_test = TUtest.drop(columns='Decision')

In [ ]:
#Simple Decision Tree
simple_tree = DecisionTreeClassifier(criterion = 'entropy',
                                    max_features = x_train.shape[1],
                                    random_state = 0,
                                    min_samples_split = 2, 
                                    min_samples_leaf = 1,
                                    max_depth = None,
                                    ccp_alpha = 0)

# Fitting the model to the training data
simple_tree.fit(x_train, y_train)

# Predict using the trained model
y_test_pred_sdt = simple_tree.predict(x_test)

print(f"Cohen's Kappa Score_test:", cohen_kappa_score(y_test, y_test_pred_sdt))

In [ ]:
#Pruned Decision Tree
path = simple_tree.cost_complexity_pruning_path(x_train, y_train)
alphas=path['ccp_alphas']
alphas

ACC_Train_simple=[]
ACC_Test_Simple=[]
for i in alphas:
    simple_tree=DecisionTreeClassifier(criterion = 'entropy',
                                    max_features = x_train.shape[1],
                                    random_state = 0,
                                    min_samples_split = 2, 
                                    min_samples_leaf = 1,
                                    max_depth = None,
                                    ccp_alpha = i)
    simple_tree.fit(x_train,y_train)
    y_train_pred=simple_tree.predict(x_train)
    y_test_pred=simple_tree.predict(x_test)
    ACC_Train_simple.append(cohen_kappa_score(y_train, y_train_pred))
    ACC_Test_Simple.append(cohen_kappa_score(y_test, y_test_pred))

alphas[ACC_Test_Simple.index(max(ACC_Test_Simple))]

In [ ]:
pruned_tree =DecisionTreeClassifier(criterion = 'entropy',
                                    max_features = x_train.shape[1],
                                    random_state = 0,
                                    min_samples_split = 2, 
                                    min_samples_leaf = 1,
                                    max_depth = None,
                                    ccp_alpha = alphas[ACC_Test_Simple.index(max(ACC_Test_Simple))])
pruned_tree.fit(x_train, y_train)

# Predict using the trained model
y_train_pred_pt = pruned_tree.predict(x_train)
y_test_pred_pt = pruned_tree.predict(x_test)

#Evaluate Model
print("Cohen's Kappa Score_train:", cohen_kappa_score(y_test, y_test_pred_pt))

In [ ]:
#Random Forest
max_features = []
best_kappa_rf = []
for i in range(1, x_train.shape[1]+1):
    RF = RandomForestClassifier(n_estimators = 100, 
                                criterion = 'entropy',
                                max_features = i, 
                                bootstrap = True,
                                max_depth= 20,
                                random_state = 0,
                                n_jobs=-1)
    RF.fit(x_train, y_train)
    y_test_pred_RF=RF.predict(x_test)
    max_features.append(i)
    best_kappa_rf.append(cohen_kappa_score(y_test, y_test_pred_RF))

# Create a data frame to store max_features and subtest_mse_rf and sort the data frame by subtest_mae_rf.
df_mae_rf = pd.DataFrame({'max_features': max_features, 'best_kappa_rf': best_kappa_rf}).sort_values(by = 'best_kappa_rf',ascending=False)
print(df_mae_rf)

In [ ]:
RF = RandomForestClassifier(n_estimators = 100, 
                           criterion = 'entropy',
                           max_features = 8, 
                           bootstrap = True,
                           max_depth= 25,
                           random_state = 0)


RF.fit(x_train, y_train)

# Predict using the fitted model
y_train_pred_RF = RF.predict(x_train)
y_test_pred_RF = RF.predict(x_test)

print("Cohen's Kappa Score_test:", cohen_kappa_score(y_test, y_test_pred_RF))

In [ ]:
PartialDependenceDisplay.from_estimator(
    RF, x_train, features=['Submit_Inquiry']
)
plt.show()

In [ ]:
feature_importances = RF.feature_importances_
feature_importance_dict = {'feature_name': x_train.columns, 'feature_importances': feature_importances}
feature_importance_df = pd.DataFrame(feature_importance_dict).sort_values(by = 'feature_importances', ascending=False)
print(feature_importance_df)

In [ ]:
# Step 1: Calculate the first and third quartile
Q1 = np.percentile(TUtrain2['Submit_FirstSource'], 25)
Q3 = np.percentile(TUtrain2['Submit_FirstSource'], 75)

# Step 2: Calculate the IQR
IQR = Q3 - Q1

# Step 3: Calculate the lower and upper bound
if Q1 - (1.5 * IQR)>=0:
    lower_bound = Q1 - (1.5 * IQR)
else:
    lower_bound=0
upper_bound = Q3 + (1.5 * IQR)

# Step 4: Print the data before removing outliers
print("Data before removing outliers:")
print(TUtrain2['Submit_FirstSource'].describe())
print(TUtest2['Submit_FirstSource'].describe())
# Step 5: Print the lower and upper bounds
print("----------------------------------------")
print("Lower Bound:", lower_bound)
print("Upper Bound:", upper_bound)
print("----------------------------------------")

# Step 6: Identify and replace the outliers
TUtrain2['Submit_FirstSource_Clean'] = TUtrain2['Submit_FirstSource'].clip(lower_bound, upper_bound)
TUtest2['Submit_FirstSource_Clean'] = TUtest2['Submit_FirstSource'].clip(lower_bound, upper_bound)
# Step 7: Print the data without outliers
print("Data after removing outliers:")
print(TUtrain2['Submit_FirstSource_Clean'].describe())
print(TUtest2['Submit_FirstSource_Clean'].describe())

In [ ]:
# Step 1: Calculate the first and third quartile
Q1 = np.percentile(TUtrain2['Submit_Inquiry'], 25)
Q3 = np.percentile(TUtrain2['Submit_Inquiry'], 75)

# Step 2: Calculate the IQR
IQR = Q3 - Q1

# Step 3: Calculate the lower and upper bound
if Q1 - (1.5 * IQR)>=0:
    lower_bound = Q1 - (1.5 * IQR)
else:
    lower_bound=0
upper_bound = Q3 + (1.5 * IQR)

# Step 4: Print the data before removing outliers
print("Data before removing outliers:")
print(TUtrain2['Submit_Inquiry'].describe())
print(TUtest2['Submit_Inquiry'].describe())
# Step 5: Print the lower and upper bounds
print("----------------------------------------")
print("Lower Bound:", lower_bound)
print("Upper Bound:", upper_bound)
print("----------------------------------------")

# Step 6: Identify and replace the outliers
TUtrain2['Submit_Inquiry_Clean'] = TUtrain2['Submit_Inquiry'].clip(lower_bound,  upper_bound)
TUtest2['Submit_Inquiry_Clean'] = TUtest2['Submit_Inquiry'].clip(lower_bound, upper_bound)
# Step 7: Print the data without outliers
print("Data after removing outliers:")
print(TUtrain2['Submit_Inquiry_Clean'].describe())
print(TUtest2['Submit_Inquiry_Clean'].describe())

In [ ]:
# Step 1: Calculate the first and third quartile
Q1 = np.percentile(TUtrain2['ACT Composite'], 25)
Q3 = np.percentile(TUtrain2['ACT Composite'], 75)

# Step 2: Calculate the IQR
IQR = Q3 - Q1

# Step 3: Calculate the lower and upper bound
if Q1 - (1.5 * IQR)>=0:
    lower_bound = Q1 - (1.5 * IQR)
else:
    lower_bound=0
upper_bound = Q3 + (1.5 * IQR)

# Step 4: Print the data before removing outliers
print("Data before removing outliers:")
print(TUtrain2['ACT Composite'].describe())
print(TUtest2['ACT Composite'].describe())
# Step 5: Print the lower and upper bounds
print("----------------------------------------")
print("Lower Bound:", lower_bound)
print("Upper Bound:", upper_bound)
print("----------------------------------------")

# Step 6: Identify and replace the outliers
TUtrain2['ACT Composite Clean'] = TUtrain2['ACT Composite'].clip(lower_bound,  upper_bound)
TUtest2['ACT Composite Clean'] = TUtest2['ACT Composite'].clip(lower_bound, upper_bound)
# Step 7: Print the data without outliers
print("Data after removing outliers:")
print(TUtrain2['ACT Composite Clean'].describe())
print(TUtest2['ACT Composite Clean'].describe())

In [ ]:
# Step 1: Calculate the first and third quartile
Q1 = np.percentile(TUtrain2['School 1 Top Percent in Class'], 25)
Q3 = np.percentile(TUtrain2['School 1 Top Percent in Class'], 75)

# Step 2: Calculate the IQR
IQR = Q3 - Q1

# Step 3: Calculate the lower and upper bound
if Q1 - (1.5 * IQR)>=0:
    lower_bound = Q1 - (1.5 * IQR)
else:
    lower_bound=0
upper_bound = Q3 + (1.5 * IQR)

# Step 4: Print the data before removing outliers
print("Data before removing outliers:")
print(TUtrain2['School 1 Top Percent in Class'].describe())
print(TUtest2['School 1 Top Percent in Class'].describe())
# Step 5: Print the lower and upper bounds
print("----------------------------------------")
print("Lower Bound:", lower_bound)
print("Upper Bound:", upper_bound)
print("----------------------------------------")

# Step 6: Identify and replace the outliers
TUtrain2['School 1 Top Percent in Class Clean'] = TUtrain2['School 1 Top Percent in Class'].clip(lower_bound,  upper_bound)
TUtest2['School 1 Top Percent in Class Clean'] = TUtest2['School 1 Top Percent in Class'].clip(lower_bound, upper_bound)
# Step 7: Print the data without outliers
print("Data after removing outliers:")
print(TUtrain2['School 1 Top Percent in Class Clean'].describe())
print(TUtest2['School 1 Top Percent in Class Clean'].describe())

In [ ]:
# Step 1: Calculate the first and third quartile
Q1 = np.percentile(TUtrain2['School 1 GPA Recalculated'], 25)
Q3 = np.percentile(TUtrain2['School 1 GPA Recalculated'], 75)

# Step 2: Calculate the IQR
IQR = Q3 - Q1

# Step 3: Calculate the lower and upper bound
if Q1 - (1.5 * IQR)>=0:
    lower_bound = Q1 - (1.5 * IQR)
else:
    lower_bound=0
upper_bound = Q3 + (1.5 * IQR)

# Step 4: Print the data before removing outliers
print("Data before removing outliers:")
print(TUtrain2['School 1 GPA Recalculated'].describe())
print(TUtest2['School 1 GPA Recalculated'].describe())
# Step 5: Print the lower and upper bounds
print("----------------------------------------")
print("Lower Bound:", lower_bound)
print("Upper Bound:", upper_bound)
print("----------------------------------------")

# Step 6: Identify and replace the outliers
TUtrain2['School 1 GPA Clean'] = TUtrain2['School 1 GPA Recalculated'].clip(lower_bound,  upper_bound)
TUtest2['School 1 GPA Clean'] = TUtest2['School 1 GPA Recalculated'].clip(lower_bound, upper_bound)
# Step 7: Print the data without outliers
print("Data after removing outliers:")
print(TUtrain2['School 1 GPA Clean'].describe())
print(TUtest2['School 1 GPA Clean'].describe())

In [ ]:

# Squared term to help the model capture curvature (U-shape)
TUtrain2['Submit_Inquiry_squared'] = TUtrain2['Submit_Inquiry'] ** 2

# Squared term to help the model capture curvature (U-shape)
TUtest2['Submit_Inquiry_squared'] = TUtest2['Submit_Inquiry'] ** 2

In [ ]:
#Drop unnecessary columns
TUtrain2.drop('Permanent Country', axis='columns', inplace=True)
TUtrain2.drop('Citizenship Status', axis='columns', inplace=True)
TUtrain2.drop('Sport 2 Sport', axis='columns', inplace=True)
TUtrain2.drop('Sport 3 Sport', axis='columns', inplace=True)
TUtrain2.drop('train-test', axis='columns', inplace=True)
TUtrain2.drop('Sport 1 Rating', axis='columns', inplace=True)
TUtrain2.drop('Entry Term (Application)', axis='columns', inplace=True)

TUtest2.drop('Permanent Country', axis='columns', inplace=True)
TUtest2.drop('Citizenship Status', axis='columns', inplace=True)
TUtest2.drop('Sport 2 Sport', axis='columns', inplace=True)
TUtest2.drop('Sport 3 Sport', axis='columns', inplace=True)
TUtest2.drop('train-test', axis='columns', inplace=True)
TUtest2.drop('Sport 1 Rating', axis='columns', inplace=True)
TUtest2.drop('Entry Term (Application)', axis='columns', inplace=True)

In [ ]:
TUtrain2.info()

In [ ]:
TUtrain2 = encode(TUtrain2)
TUtest2= encode(TUtest2)
y_train2= TUtrain2['Decision']
x_train2 = TUtrain2.drop(columns='Decision')
y_test2=TUtest2['Decision']
x_test2 = TUtest2.drop(columns='Decision')

In [ ]:
#Refined Random Forest
max_features2 = []
best_kappa_rf2 = []
for i in range(1, x_train2.shape[1]+1):
    RF2 = RandomForestClassifier(n_estimators = 200, 
                                criterion = 'entropy',
                                max_features = i, 
                                bootstrap = True,
                                max_depth=20,
                                random_state = 0,
                                n_jobs=-1)
    RF2.fit(x_train2, y_train2)
    y_test_pred_RF2=RF2.predict(x_test2)
    max_features2.append(i)
    best_kappa_rf2.append(cohen_kappa_score(y_test2, y_test_pred_RF2))

# Create a data frame to store max_features and subtest_mse_rf and sort the data frame by subtest_mae_rf.
df_mae_rf2 = pd.DataFrame({'max_features': max_features2, 'best_kappa_rf': best_kappa_rf2}).sort_values(by = 'best_kappa_rf',ascending=False)
print(df_mae_rf2)

In [ ]:
RF2 = RandomForestClassifier(n_estimators = 200, 
                                criterion = 'entropy',
                                max_features = 10, 
                                bootstrap = True,
                                max_depth=20,
                                random_state = 0,
                                n_jobs=-1)
RF2.fit(x_train2, y_train2)
y_test_pred_RF2=RF2.predict(x_test2)
print("Cohen's Kappa Score_test:", cohen_kappa_score(y_test2, y_test_pred_RF2))

In [ ]:
feature_importances2 = RF2.feature_importances_
feature_importance_dict2 = {'feature_name': x_train2.columns, 'feature_importances': feature_importances2}
feature_importance_df2 = pd.DataFrame(feature_importance_dict2).sort_values(by = 'feature_importances', ascending=False)
print(feature_importance_df2)